In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import os

df_path = os.path.join(path,"Q1_data.csv")
df = pd.read_csv(df_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns=["Order_ID"])

In [ ]:
# Task 2: Write your code here:
missing_percentage = (df.isnull().sum() / len(df)) * 100 # get the percentage of missing values in each column
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False) # remove columns wih no missing values

print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
#Task 2 continue
# Remove nulls form the target values and because we should not fill it with any thing
# Remove nulls form the all missing columns and because its too small missing ro
df_clean = df.dropna(subset=["Delivery_Time","Weather","Traffic_Level","Time_of_Day","Courier_Experience_yrs"])

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import LabelEncoder
categorical_cols = df_clean.select_dtypes(include=["object"]).columns

#initlizing
for col in categorical_cols:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

df_clean.info()

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import  StandardScaler

scaler = StandardScaler()
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time") # will include every col except the target
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean

In [ ]:
# Task 6: Write your code here:plt.figure(figsize=(10, 5))
plt.figure(figsize=(10, 5))
plt.hist(df_clean['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
from sklearn.model_selection import  train_test_split
X = df_clean.drop("Delivery_Time",axis=1)
y = df_clean['Delivery_Time']


X.head()

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import numpy as np

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
mae_scores = []

# we use KFold because its regressing problem
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
    # indexing for each fold
    X_train,  X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae_scores.append(mean_absolute_error(y_test, y_pred))
mae_scores = np.array(mae_scores)
print(f"5-Fold CV Results:")
print(f"MAE:  {mae_scores.mean():,.2f}")

In [ ]:
# Task 1: Write your code here:
# Feature importance

feature_cols = ["Distance_km",	"Weather",	"Traffic_Level",	"Time_of_Day",	"Vehicle_Type",	"Preparation_Time_min",	"Courier_Experience_yrs"]
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: